# AgentCore Evaluations Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## 🎯 What You'll Build

In this lab you will set up a **complete evaluation pipeline** for a Strands AI agent using **Amazon Bedrock AgentCore Evaluations**.

By the end of this lab, you will have:
- ✅ Deployed a Strands agent to AgentCore Runtime with automatic OTEL instrumentation
- ✅ Explored all 13 built-in evaluators
- ✅ Created a custom LLM-as-judge evaluator with a 5-level scoring scale
- ✅ Run **on-demand evaluations** at session, trace, and span levels
- ✅ Interpreted evaluation results (label, value, explanation)
- ✅ Configured **online evaluation** for continuous production monitoring
- ✅ Saved evaluation results to a JSON file

---

## 🏗️ Architecture

```
┌──────────────────────────────────────────────┐
│           Strands Agent                      │
│   (Claude Haiku 4.5 + Math + Weather tools)  │
└──────────────┬───────────────────────────────┘
               │  Deployed to AgentCore Runtime
               ▼
┌──────────────────────────────────────────────┐
│      AgentCore Runtime                       │
│  (auto OTEL via AgentCore Runtime)    │
└──────────────┬───────────────────────────────┘
               │  OTEL Traces
               ▼
┌──────────────────────────────────────────────┐
│    AgentCore Observability + CloudWatch      │
│  Sessions → Traces → Spans (Tool Calls)      │
└──────────────┬───────────────────────────────┘
               │
      ┌────────┴────────┐
      ▼                 ▼
 On-Demand          Online Eval
 Evaluation         Configuration
 (developer         (continuous
  triggered)         sampling)
```

---


## ✅ Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to Amazon Bedrock
- Bedrock access enabled for: `us.anthropic.claude-haiku-4-5-20251001-v1:0`
- Access to ECR (for container deployment)

---
# Part 1: Environment Setup

In [ ]:
# Install required packages
import shutil
import subprocess
import sys

packages = [
    "bedrock-agentcore",
    "bedrock-agentcore-starter-toolkit>=0.3.4",
    "boto3>=1.42.80",
    "pickleshare",
    "strands-agents",
    "strands-agents-tools",
    # AgentCore Runtime already emits OTEL traces for deployed agents
]

if shutil.which("uv"):
    subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, *packages], stdout=subprocess.DEVNULL)
    print("✅ All packages installed via uv")
else:
    try:
        import pip  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pkg, "-q"])

    print("✅ All packages installed via pip fallback")

In [ ]:
import os

os.environ['AWS_REGION'] = 'us-east-1'

# APPROACH A: Use credentials
# os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
# os.environ['AWS_SESSION_TOKEN'] = "your_session_token"

# APPROACH B: Use AWS SSO profile
#os.environ['AWS_PROFILE'] = 'your_profile'

# Remove any existing credential env vars to force profile usage
#for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
#    os.environ.pop(key, None)

os.environ['AWS_REGION'] = 'us-east-1'

print("✅ AWS Profile set. Please restart kernel and run all cells.")

In [ ]:
import boto3
import json
import os
import uuid
import time
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display

boto_session = Session()
region = boto_session.region_name or "us-east-1"

try:
    identity = boto3.client("sts").get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   ARN:     {identity['Arn']}")
    print(f"   Region:  {region}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")

---
# Part 2: Deploy a Strands Agent to AgentCore Runtime

We'll deploy a simple **weather + math assistant** to AgentCore Runtime.

This agent is intentionally simple so we can test evaluation metrics effectively:
- ✅ Math questions → should answer correctly
- ✅ Weather questions → should answer correctly (using weather tool)
- ❌ Out-of-scope questions → our custom evaluator should penalize these

> **Important**: AgentCore Runtime already emits OTEL traces for deployed agents — no extra OTEL package is required here.

In [ ]:
# Write the agent entry point file
def get_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "backend").exists() and cwd.name == "capstone_project":
        return cwd
    if (cwd / "capstone_project" / "backend").exists():
        return cwd / "capstone_project"
    if cwd.name == "notebooks" and (cwd.parent / "backend").exists():
        return cwd.parent
    raise FileNotFoundError("Could not locate the capstone_project backend directory from the current working directory.")

PROJECT_DIR = get_project_root()
EVALUATION_DIR = PROJECT_DIR / "backend" / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
AGENT_FILE = EVALUATION_DIR / "eval_agent_strands.py"
REQUIREMENTS_FILE = EVALUATION_DIR / "requirements_eval.txt"

AGENT_CODE = '''from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands_tools import calculator
import json

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

@tool
def get_weather(location: str) -> dict:
    """Get current weather for a location. Returns a mock response for this demo."""
    # Simplified mock — real implementation would call a weather API
    weather_data = {
        "location": location,
        "temperature": "22°C",
        "condition": "Partly cloudy",
        "humidity": "65%"
    }
    return weather_data

app = BedrockAgentCoreApp()

model = BedrockModel(model_id=MODEL_ID)

agent = Agent(
    model=model,
    tools=[calculator, get_weather],
    system_prompt=(
        "You are a helpful assistant. You can perform math calculations "
        "and check the weather. Stay focused on these topics only."
    )
)

@app.entrypoint
def invoke_agent(payload):
    prompt = payload.get("prompt", "")
    result = agent(prompt)
    return str(result)

if __name__ == "__main__":
    app.run()
'''

REQUIREMENTS = """bedrock-agentcore
strands-agents
strands-agents-tools
boto3
"""

# Save to files for deployment
with open(AGENT_FILE, "w") as f:
    f.write(AGENT_CODE)

with open(REQUIREMENTS_FILE, "w") as f:
    f.write(REQUIREMENTS)

print(f"✅ Agent code written to: {AGENT_FILE}")
print(f"✅ Requirements written to: {REQUIREMENTS_FILE}")
print()
print("Agent capabilities:")
print("  🧮 calculator — math operations")
print("  🌤️  get_weather — weather lookups")
print("  ❌ Out-of-scope — custom evaluator will penalize these")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

RUNTIME_WORK_DIR = EVALUATION_DIR

def wait_for_runtime_name_to_clear(agent_name, region, attempts=20, delay_seconds=15):
    """Wait until no runtime with this logical name is stuck in DELETING."""
    ctrl = boto3.client("bedrock-agentcore-control", region_name=region)
    for attempt in range(1, attempts + 1):
        matching = [
            item for item in ctrl.list_agent_runtimes(maxResults=100).get("agentRuntimes", [])
            if item.get("agentRuntimeName") == agent_name
        ]
        deleting = [item for item in matching if item.get("status") == "DELETING"]
        if not deleting:
            return
        if attempt == attempts:
            ids = ", ".join(item.get("agentRuntimeId", "?") for item in deleting)
            raise RuntimeError(f"Runtime name {agent_name} is still deleting after waiting: {ids}")
        ids = ", ".join(item.get("agentRuntimeId", "?") for item in deleting)
        print(f"⏳ Existing runtime still deleting ({ids}). Waiting {delay_seconds}s...")
        time.sleep(delay_seconds)

stale_runtime_config = RUNTIME_WORK_DIR / ".bedrock_agentcore.yaml"
if os.path.exists(stale_runtime_config):
    os.remove(stale_runtime_config)
    print(f"🧹 Removed stale {stale_runtime_config} so deployment starts cleanly")

agentcore_runtime = Runtime()

AGENT_NAME = "eval_lab_agent"

wait_for_runtime_name_to_clear(AGENT_NAME, region)

os.chdir(RUNTIME_WORK_DIR)
print(f"📁 Runtime work directory: {RUNTIME_WORK_DIR}")

print(f"🚀 Deploying agent: {AGENT_NAME}")
print("   This builds a Docker container and deploys to AgentCore Runtime.")
print("   ⏱️  This takes ~10-15 minutes. Continue reading while it runs!\n")

agentcore_runtime.configure(
    entrypoint=AGENT_FILE.name,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=REQUIREMENTS_FILE.name,
    region=region,
    agent_name=AGENT_NAME,
    idle_timeout=120
)

launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print(f"\n   Deployment initiated: {launch_result}")

# Store for persistence across cells
%store launch_result

In [ ]:
# Wait for deployment to become READY
print("⏳ Waiting for agent to reach READY status...")
print("   (Check the AgentCore console for live status)\n")

end_statuses = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while True:
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"   Status: {status}")
    if status in end_statuses:
        break
    time.sleep(15)

if status == "READY":
    print(f"\n✅ Agent deployed successfully!")
    print(f"   Agent ID:  {launch_result.agent_id}")
    print(f"   Agent ARN: {launch_result.agent_arn}")
else:
    print(f"\n❌ Deployment failed with status: {status}")
    print("   Check the AgentCore console for error details.")

In [ ]:
# Invoke the agent to generate traces
session_id = str(uuid.uuid4())
print(f"📡 Session ID: {session_id}")
print("   (Save this — we use it for on-demand evaluations)\n")

# Three test invocations covering different scenarios
test_prompts = [
    ("Math (in-scope)",       "How much is 2 + 2?"),
    ("Weather (in-scope)",    "What is the weather like today?"),
    ("Capital (OUT-OF-SCOPE)","Can you tell me the capital of the United States?"),
]

for label, prompt in test_prompts:
    print(f"   🔹 {label}")
    print(f"      Prompt: {prompt}")
    response = agentcore_runtime.invoke(
        payload={"prompt": prompt},
        session_id=session_id
    )
    print(f"      Response: {str(response)[:120]}...")
    print()

print("\n✅ Agent invocations complete — traces are being generated in AgentCore Observability!")
print("   Waiting 30 seconds before starting the span-ingestion check...")
time.sleep(30)
print("   ✅ Initial wait complete.")

# Store session_id for use in later cells
%store session_id

---
# Part 3: Explore Built-In Evaluators

AgentCore provides **13 pre-configured evaluators** ready to use immediately.
Let's explore them before running any evaluations.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation(region=region)

print("📋 All Available Built-In Evaluators:\n")

available = eval_client.list_evaluators()
evaluators = available.get("evaluators", [])

# Group by inferred category
categories = {
    "Response Quality": [],
    "Task Completion":  [],
    "Tool Level":       [],
    "Safety":          []
}

for ev in evaluators:
    name = ev.get("evaluatorId", "")
    desc = ev.get("description", "")
    if "Goal" in name:
        categories["Task Completion"].append((name, desc))
    elif "Tool" in name:
        categories["Tool Level"].append((name, desc))
    elif any(w in name for w in ["Harm", "Stereo", "Refusal", "Privacy", "Topic"]):
        categories["Safety"].append((name, desc))
    else:
        categories["Response Quality"].append((name, desc))

for category, items in categories.items():
    print(f"{'─'*50}")
    print(f"  {category}")
    print(f"{'─'*50}")
    for name, desc in items:
        print(f"  • {name}")
        print(f"    {desc[:80]}")
    print()

print(f"Total: {len(evaluators)} built-in evaluators")

In [ ]:
# Deep-dive into Builtin.Correctness
print("🔍 Deep-dive: Builtin.Correctness\n")

correctness_detail = eval_client.get_evaluator(evaluator_id="Builtin.Correctness")

print(json.dumps(correctness_detail, indent=2, default=str))

print()
print("💡 Key Observations:")
print("   • This is an LLM-as-judge evaluator")
print("   • It operates at TRACE level (one score per user turn)")
print("   • The configuration is fixed — you cannot modify it")
print("   • That fixed config ensures consistency across all evaluations")
print("   • Notice the rating scale — only 3 levels: Incorrect/Partially/Correct")
print("   ➡️  We'll create a custom 5-level version in Part 4")

---
# Part 4: Create a Custom Evaluator

Built-in `Correctness` only has 3 levels. We want a **5-level scale** that also penalizes out-of-scope answers.

Our custom evaluator will:
- Use **5 levels**: Very Good → Good → OK → Poor → Very Poor
- Penalize agents that answer questions **outside their scope** (weather + math only)
- Operate at **TRACE level** (per user turn)

In [ ]:
# Define the custom evaluator configuration
custom_eval_config = {
    "llmAsAJudge": {
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
                "inferenceConfig": {
                    "maxTokens": 500,
                    "temperature": 1.0
                }
            }
        },
        "instructions": (
            "You are evaluating the quality of an AI assistant's response.\n"
            "You are given a task and the assistant's candidate response.\n\n"
            "Evaluate whether the response accurately addresses the question.\n"
            "Consider factual accuracy, completeness, and clarity.\n\n"
            "**IMPORTANT SCOPE RULE**: This assistant is ONLY supposed to answer "
            "questions about weather and mathematical calculations. "
            "If the assistant answers questions outside this scope "
            "(e.g., geography, history, general knowledge), "
            "it MUST receive a 'Very Poor' rating regardless of the response quality.\n\n"
            "Context: {context}\n"
            "Candidate Response: {assistant_turn}"
        ),
        "ratingScale": {
            "numerical": [
                {
                    "value": 1.0,
                    "label": "Very Good",
                    "definition": (
                        "Response is completely accurate, directly answers the question, "
                        "and stays within the agent's defined scope (weather or math)."
                    )
                },
                {
                    "value": 0.75,
                    "label": "Good",
                    "definition": (
                        "Response is mostly accurate with minor issues. Core answer is "
                        "correct. Stays within scope."
                    )
                },
                {
                    "value": 0.50,
                    "label": "OK",
                    "definition": (
                        "Response is partially correct but contains notable errors or "
                        "incomplete information. Within scope but imperfect."
                    )
                },
                {
                    "value": 0.25,
                    "label": "Poor",
                    "definition": (
                        "Response contains significant errors or misconceptions. "
                        "Mostly incorrect but within scope."
                    )
                },
                {
                    "value": 0.0,
                    "label": "Very Poor",
                    "definition": (
                        "Response is completely incorrect, OR the agent answered a "
                        "question outside its defined scope (non-weather, non-math). "
                        "Scope violations always result in Very Poor regardless of answer quality."
                    )
                }
            ]
        }
    }
}

print("📊 Custom Evaluator Configuration:")
print(f"   Model: {custom_eval_config['llmAsAJudge']['modelConfig']['bedrockEvaluatorModelConfig']['modelId']}")
print(f"   Level: TRACE (one result per user turn)")
print(f"   Scale: 5 levels (0.0 Very Poor → 1.0 Very Good)")
print(f"   Scope rule: Out-of-scope answers → Very Poor")

In [ ]:
print("🔧 Creating custom evaluator...\n")

custom_evaluator = eval_client.create_evaluator(
    name="response_quality_with_scope",
    level="TRACE",
    description=(
        "5-level response quality evaluator that penalizes out-of-scope answers. "
        "Built for weather+math agents."
    ),
    config=custom_eval_config
)

evaluator_id = custom_evaluator["evaluatorId"]

print(f"✅ Custom Evaluator Created!")
print(f"   Evaluator ID: {evaluator_id}")
print(f"   Name: response_quality_with_scope")
print(f"   Level: TRACE")
print(f"   Scale: Very Poor (0.0) → Very Good (1.0)")

# Store for later use
%store evaluator_id

---
# Part 5: On-Demand Evaluations

Now let's evaluate the agent session we created in Part 2.

We'll run evaluations at all three levels:
- **Session level**: Goal Success Rate (whole conversation)
- **Trace level**: Correctness + our custom evaluator (per turn)
- **Span level**: Tool Selection + Parameter Accuracy (per tool call)

In [ ]:
# Restore stored variables
%store -r launch_result
%store -r session_id
%store -r evaluator_id

print("📂 Loaded evaluation context:")
print(f"   Agent ID:     {launch_result.agent_id}")
print(f"   Agent ARN:    {launch_result.agent_arn}")
print(f"   Session ID:   {session_id}")
print(f"   Custom Eval:  {evaluator_id}")

from datetime import datetime, timedelta
from bedrock_agentcore_starter_toolkit.operations.observability.client import ObservabilityClient

def wait_for_session_spans(agent_id, session_id, region, attempts=12, delay_seconds=20, lookback_days=1):
    """Block until the session spans are visible in AgentCore Observability."""
    obs_client = ObservabilityClient(region_name=region)
    for attempt in range(1, attempts + 1):
        end_time = datetime.now()
        start_time = end_time - timedelta(days=lookback_days)
        spans = obs_client.query_spans_by_session(
            session_id=session_id,
            start_time_ms=int(start_time.timestamp() * 1000),
            end_time_ms=int(end_time.timestamp() * 1000),
            agent_id=agent_id,
        )
        if spans:
            print(f"   Found {len(spans)} spans for session {session_id}.")
            return spans
        if attempt == attempts:
            raise RuntimeError(
                f"No spans found for session {session_id} after {attempts} checks. "
                "Check CloudWatch / AgentCore Observability ingestion."
            )
        print(f"   Spans not ready yet (attempt {attempt}/{attempts}). Waiting {delay_seconds}s...")
        time.sleep(delay_seconds)

def run_evaluation_with_retry(eval_client, agent_id, session_id, evaluators):
    wait_for_session_spans(agent_id=agent_id, session_id=session_id, region=region)
    return eval_client.run(
        agent_id=agent_id,
        session_id=session_id,
        evaluators=evaluators,
    )


In [ ]:
print("🎯 SESSION-LEVEL: Goal Success Rate\n")
print("Question: Did the agent complete all the user's goals across the full conversation?")
print()

goal_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.GoalSuccessRate"]
)

print(f"Results ({len(goal_results.results)} result for the whole session):\n")
for result in goal_results.results:
    display(Markdown(f"""
**Evaluator**: `{result.evaluator_name}`

**Score**: `{result.label}` ({result.value})

**Explanation**: {result.explanation}

**Tokens Used**: {result.token_usage}
"""))
    print("─" * 60)

In [ ]:
print("✅ TRACE-LEVEL: Correctness\n")
print("Question: Is each individual agent response factually correct?")
print("(One result per user turn)")
print()

correctness_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness"]
)

print(f"Results ({len(correctness_results.results)} results — one per turn):\n")
for i, result in enumerate(correctness_results.results, 1):
    display(Markdown(f"""
**Turn {i}** | **{result.evaluator_name}**

**Score**: `{result.label}` ({result.value})

**Explanation**: {result.explanation}
"""))
    print("─" * 60)

In [ ]:
print("🔧 SPAN-LEVEL: Tool Selection & Parameter Accuracy\n")
print("Question: Did the agent choose the right tools with the right parameters?")
print("(One result per tool call — multiple possible per turn)")
print()

tool_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[
        "Builtin.ToolSelectionAccuracy",
        "Builtin.ToolParameterAccuracy"
    ]
)

print(f"Results ({len(tool_results.results)} results — one per tool invocation per metric):\n")
for result in tool_results.results:
    display(Markdown(f"""
**Metric**: `{result.evaluator_name}`

**Score**: `{result.label}` ({result.value})

**Explanation**: {result.explanation}

**Context**: {str(result.context)[:200]}
"""))
    print("─" * 60)

In [ ]:
print("🎨 TRACE-LEVEL: Custom Evaluator (5-level + scope enforcement)\n")
print("Question: Is the response high quality AND does the agent stay in scope?")
print()
print("Expected behavior:")
print("  • Math turn (2+2)    → Very Good or Good (correct, in-scope)")
print("  • Weather turn       → Good or OK (tool used correctly, in-scope)")
print("  • Capital turn       → Very Poor (out-of-scope! capital is not weather or math)")
print()

custom_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[evaluator_id]
)

print(f"Results ({len(custom_results.results)} results — one per turn):\n")
for i, result in enumerate(custom_results.results, 1):
    score_value = result.value if result.value is not None else 0
    score_text = result.value if result.value is not None else "n/a"
    score_emoji = "✅" if score_value >= 0.75 else ("⚠️" if score_value >= 0.5 else "❌")
    display(Markdown(f"""
{score_emoji} **Turn {i}** | `{result.evaluator_name}`

**Score**: `{result.label}` ({score_text})

**Explanation**: {result.explanation}
"""))
    print("─" * 60)

print()
print("💡 Did the third turn get 'Very Poor'?")
print("   That's our custom scope rule kicking in — the agent answered geography, not math/weather.")

In [ ]:
# Save evaluation results to a JSON file
print("💾 Saving evaluation results to file...\n")

os.makedirs("eval_results", exist_ok=True)

save_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        evaluator_id
    ],
    output="eval_results/lab_evaluation_output.json"
)

# Check file was saved
if os.path.exists("eval_results/lab_evaluation_output.json"):
    with open("eval_results/lab_evaluation_output.json") as f:
        saved = json.load(f)
    print(f"✅ Results saved to: eval_results/lab_evaluation_output.json")
else:
    print("ℹ️  Results may have been saved to a different location.")
    print("   Check the eval_client.run() output above for the file path.")

---
# Part 6: Online Evaluation — Continuous Production Monitoring

Now let's configure **online evaluation** — this runs automatically against live traffic.

Online evaluation is the production monitoring mode. Once configured, it evaluates your agent continuously based on the sampling rate you define.

In [ ]:
print("⚙️ Creating Online Evaluation Configuration...\n")

online_response = eval_client.create_online_config(
    agent_id=launch_result.agent_id,
    config_name="pumping_code_quality_monitor",
    sampling_rate=100,   # 100% for this lab; use 10-20% in production
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolParameterAccuracy",
        "Builtin.ToolSelectionAccuracy",
        evaluator_id  # our custom scope evaluator
    ],
    config_description=(
        "Continuous quality monitoring for weather+math assistant. "
        "Checks correctness, tool accuracy, and scope adherence."
    ),
    auto_create_execution_role=True
)

online_config_id = online_response["onlineEvaluationConfigId"]

print(f"✅ Online Evaluation Config Created!")
print(f"   Config ID: {online_config_id}")
print(f"   Sampling Rate: 100% (all sessions)")
print(f"   Evaluators: 5 (4 built-in + 1 custom)")
print(f"   Status: ENABLED — evaluating live traffic automatically")

%store online_config_id

In [ ]:
# Verify the config is active
config_details = eval_client.get_online_config(config_id=online_config_id)

print("📋 Online Evaluation Config Details:")
print(json.dumps(config_details, indent=2, default=str))

In [ ]:
# Invoke the agent multiple times to trigger online evaluation
agentcore_data_client = boto3.client("bedrock-agentcore", region_name=region)

def invoke_agent_direct(agent_arn, prompt):
    """Invoke the deployed agent via boto3."""
    response = agentcore_data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt})
    )
    events = []
    for event in response.get("response", []):
        if hasattr(event, "read"):
            events.append(event.read().decode("utf-8"))
        elif isinstance(event, bytes):
            events.append(event.decode("utf-8"))
    return events[0] if events else "{}"

print("📡 Triggering new agent sessions to activate online evaluation...\n")
print("   These new sessions will be automatically evaluated by our online config.")
print()

online_prompts = [
    "How much is 7 + 9 + 10 * 2?",
    "Is it raining right now?",
    "What is 20% of 300?",
    "What can you help me with?",
    "What is the capital of New York State?"  # out-of-scope!
]

for prompt in online_prompts:
    print(f"   → {prompt}")
    try:
        result = invoke_agent_direct(launch_result.agent_arn, prompt)
        print(f"     ✅ Response received")
    except Exception as e:
        print(f"     ⚠️  {str(e)[:60]}")

print()
print("✅ Agent invocations complete!")
print("   Online evaluation is now processing these sessions in the background.")
print("   Results appear in CloudWatch under AgentCore Observability.")
print()
print("📊 Where to view results:")
print("   AWS Console → CloudWatch → Gen AI Observability → AgentCore → Agents")
print("   → Select your agent → DEFAULT endpoint → Evaluation tab")
print("   (May take 2-5 minutes to appear)")

---
# 🧹 Cleanup

Clean up all resources to avoid ongoing costs.

**Note**: The deployed AgentCore Runtime agent will also stop incurring costs when idle (after the `idle_timeout` period).

In [ ]:
print("🧹 Starting cleanup...\n")

# ── Step 1: Delete online evaluation config ───────────────────────────────
try:
    print("Step 1: Deleting online evaluation config...")
    %store -r online_config_id
    eval_client.delete_online_config(config_id=online_config_id)
    print(f"   ✅ Online config deleted: {online_config_id}")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 2: Delete custom evaluator ──────────────────────────────────────
try:
    print("\nStep 2: Deleting custom evaluator...")
    %store -r evaluator_id
    eval_client.delete_evaluator(evaluator_id=evaluator_id)
    print(f"   ✅ Custom evaluator deleted: {evaluator_id}")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 3: Delete the deployed agent ────────────────────────────────────
try:
    print("\nStep 3: Deleting AgentCore Runtime agent...")
    %store -r launch_result
    agent_id = getattr(launch_result, "agent_id", None)
    ctrl = boto3.client("bedrock-agentcore-control", region_name=region)

    if agent_id:
        ctrl.delete_agent_runtime(agentRuntimeId=agent_id)
        print(f"   Delete requested for agent runtime: {agent_id}")
    else:
        raise RuntimeError("launch_result.agent_id is missing")

    for attempt in range(1, 21):
        matching = [
            item for item in ctrl.list_agent_runtimes(maxResults=100).get("agentRuntimes", [])
            if item.get("agentRuntimeId") == agent_id
        ]
        if not matching:
            print(f"   ✅ Agent {agent_id} deleted")
            break
        status = matching[0].get("status")
        print(f"   Waiting for deletion... current status: {status}")
        time.sleep(10)
    else:
        print(f"   ⚠️  Agent {agent_id} still exists after waiting. Status: {matching[0].get('status')}")
except Exception as e:
    print(f"   ⚠️  {str(e)[:120]}")
    print("   You can also delete from the AgentCore console manually.")

# ── Step 4: Clean up local files ─────────────────────────────────────────
import shutil
for fname in [AGENT_FILE, REQUIREMENTS_FILE, RUNTIME_WORK_DIR / ".bedrock_agentcore.yaml"]:
    if os.path.exists(fname):
        os.remove(fname)

print("\n🎉 Cleanup complete!")
print("   eval_results/ folder retained — check lab_evaluation_output.json for your results.")

---
# 🎉 Lab Complete!

## What You Accomplished

| Step | Concept | What You Learned |
|------|---------|------------------|
| Part 2 | Deploy agent to Runtime | OTEL instrumentation is automatic |
| Part 3 | Explore built-in evaluators | 13 pre-built metrics, fixed configs |
| Part 4 | Custom evaluator (5-level) | LLM-as-judge with scope enforcement |
| Part 5 | On-demand: GoalSuccessRate | Session-level task completion |
| Part 5 | On-demand: Correctness | Trace-level per-turn accuracy |
| Part 5 | On-demand: Tool metrics | Span-level tool usage quality |
| Part 5 | On-demand: Custom | Scope penalization in action |
| Part 6 | Online evaluation | Continuous automatic monitoring |
| Part 7 | Results analysis | How to read and act on scores |

## Key Takeaways

- 📊 **Three levels**: Span (tool calls) → Trace (turns) → Session (full conversation)
- 🤖 **13 built-in evaluators** cover most use cases immediately
- 🎨 **Custom evaluators** enable domain-specific and scope-aware quality checks
- 💡 **The explanation field** is the most valuable field — read it to understand WHY
- 🔄 **On-demand + Online** complement each other: dev-time testing + prod monitoring
